# Camada Silver — Processo Sicredi

Pipeline da Camada Prata (Silver) lendo as tabelas Delta da camada Bronze (`workspace.bronze`) e gravando no schema `workspace.silver`, conforme as regras do desafio:

- **Tipagem correta** dos campos (IDs `long`, datas `date`, `valor` `decimal(18,2)`).
- **Deduplicação** por chave de negócio, priorizando a `data_atualizacao` mais recente (clientes e cartões) ou ordenação determinística quando não há `data_atualizacao`.
- **Versão mais recente** apenas do cadastro de cliente e cartão.
- **Quarentena**: registros inválidos vão para tabelas `silver.quarentena_*` com coluna `motivo_erro`.
- **Integridade referencial** validada em cascata: cliente → conta → cartão → transação → estorno.
- **Transações estornadas** identificadas via `flag_estornada` e excluídas da tabela `silver.transacoes_financeiras`, base para os indicadores da camada Gold.

In [0]:
# ============================================================
# Parâmetros do pipeline Silver
# ============================================================
from pyspark.sql import functions as F, Window

CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

def salvar_quarentena(df_avaliado, colunas, tabela):
    """Grava os registros inválidos (motivo_erro preenchido) na tabela de
    quarentena, com todas as colunas originais como STRING + motivo_erro."""
    q = df_avaliado.filter(F.col("motivo_erro") != "").select(
        *[F.col(c).cast("string").alias(c) for c in colunas],
        F.col("motivo_erro"),
        F.current_timestamp().alias("timestamp_processamento"),
    )
    q.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    n = q.count()
    print(f"  Quarentena {tabela}: {n} registros")
    return n

def publicar_silver(validos, colunas, tabela, chave, ordenacao):
    """Deduplica os registros válidos pela chave de negócio mantendo a
    versão mais recente (ordenacao) e grava na tabela Silver."""
    w = Window.partitionBy(chave).orderBy(*ordenacao)
    dedup = (
        validos
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
        .withColumn("timestamp_processamento", F.current_timestamp())
    )
    saida = dedup.select(*colunas, "timestamp_processamento")
    saida.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    n = saida.count()
    print(f"  Silver {tabela}: {n} registros (dedup por {chave})")
    return n

print("Parâmetros configurados:", BRONZE, "->", SILVER)

In [0]:
# ============================================================
# 1. CLIENTES — tipagem, validação, quarentena e dedup
# Regra: manter apenas a versão mais recente do cadastro
#        (maior data_atualizacao por id_cliente).
# ============================================================
COLUNAS_CLIENTES = ["id_cliente", "nome", "cpf", "data_nascimento", "cidade", "estado", "data_atualizacao"]

clientes_tipado = spark.table(f"{BRONZE}.clientes").select(
    F.col("id_cliente").cast("long").alias("id_cliente"),
    F.col("nome").cast("string").alias("nome"),
    F.col("cpf").cast("string").alias("cpf"),
    F.expr("try_cast(data_nascimento as date)").alias("data_nascimento"),
    F.col("cidade").cast("string").alias("cidade"),
    F.col("estado").cast("string").alias("estado"),
    F.expr("try_cast(data_atualizacao as date)").alias("data_atualizacao"),
)

clientes_avaliado = clientes_tipado.withColumn(
    "motivo_erro",
    F.concat_ws(
        " | ",
        F.when(F.col("id_cliente").isNull(), F.lit("chave_nula:id_cliente")),
        F.when(~F.col("cpf").rlike("^\\d{11}$"), F.lit("cpf_invalido")),
        F.when(F.col("data_nascimento").isNull(), F.lit("data_nascimento_invalida")),
        F.when(F.col("data_nascimento") > F.current_date(), F.lit("data_nascimento_futura")),
    ),
)

clientes_validos = clientes_avaliado.filter(F.col("motivo_erro") == "")

qtd_q_clientes = salvar_quarentena(
    clientes_avaliado, COLUNAS_CLIENTES, f"{SILVER}.quarentena_clientes"
)
qtd_s_clientes = publicar_silver(
    clientes_validos,
    COLUNAS_CLIENTES,
    f"{SILVER}.clientes",
    "id_cliente",
    [F.col("data_atualizacao").desc(), F.col("cpf").desc()],
)
print(f"CLIENTES: bronze={clientes_tipado.count()} | silver={qtd_s_clientes} | quarentena={qtd_q_clientes}")

In [0]:
# ============================================================
# 2. CONTAS — tipagem, validação e integridade referencial
# Regra: id_cliente deve existir em silver.clientes (cascata).
# ============================================================
COLUNAS_CONTAS = ["id_conta", "id_cliente", "tipo_conta", "status_conta", "data_abertura"]

contas_tipado = spark.table(f"{BRONZE}.contas").select(
    F.col("id_conta").cast("long").alias("id_conta"),
    F.col("id_cliente").cast("long").alias("id_cliente"),
    F.col("tipo_conta").cast("string").alias("tipo_conta"),
    F.col("status_conta").cast("string").alias("status_conta"),
    F.expr("try_cast(data_abertura as date)").alias("data_abertura"),
)

clientes_silver = spark.table(f"{SILVER}.clientes").select(
    F.col("id_cliente"), F.lit(True).alias("_ri_ok")
)

contas_com_ri = contas_tipado.join(clientes_silver, "id_cliente", "left")

contas_avaliado = contas_com_ri.withColumn(
    "motivo_erro",
    F.concat_ws(
        " | ",
        F.when(F.col("id_conta").isNull(), F.lit("chave_nula:id_conta")),
        F.when(F.col("id_cliente").isNull(), F.lit("chave_nula:id_cliente")),
        F.when(F.col("data_abertura").isNull(), F.lit("data_abertura_invalida")),
        F.when(F.col("_ri_ok").isNull(), F.lit("integridade_referencial:id_cliente_inexistente")),
    ),
).select(*[F.col(c) for c in COLUNAS_CONTAS], F.col("motivo_erro"))

contas_validos = contas_avaliado.filter(F.col("motivo_erro") == "")

qtd_q_contas = salvar_quarentena(contas_avaliado, COLUNAS_CONTAS, f"{SILVER}.quarentena_contas")
qtd_s_contas = publicar_silver(
    contas_validos,
    COLUNAS_CONTAS,
    f"{SILVER}.contas",
    "id_conta",
    [F.col("data_abertura").desc(), F.col("id_cliente").desc()],
)
print(f"CONTAS: bronze={contas_tipado.count()} | silver={qtd_s_contas} | quarentena={qtd_q_contas}")

In [0]:
# ============================================================
# 3. CARTÕES — tipagem, validação, RI e versão mais recente
# Regra: id_conta deve existir em silver.contas (cascata) e
#        manter apenas a versão com data_atualizacao mais recente.
# ============================================================
COLUNAS_CARTOES = ["id_cartao", "id_conta", "bandeira", "tipo_cartao", "status_cartao", "data_emissao", "data_atualizacao"]

cartoes_tipado = spark.table(f"{BRONZE}.cartoes").select(
    F.col("id_cartao").cast("long").alias("id_cartao"),
    F.col("id_conta").cast("long").alias("id_conta"),
    F.col("bandeira").cast("string").alias("bandeira"),
    F.col("tipo_cartao").cast("string").alias("tipo_cartao"),
    F.col("status_cartao").cast("string").alias("status_cartao"),
    F.expr("try_cast(data_emissao as date)").alias("data_emissao"),
    F.expr("try_cast(data_atualizacao as date)").alias("data_atualizacao"),
)

contas_silver = spark.table(f"{SILVER}.contas").select(
    F.col("id_conta"), F.lit(True).alias("_ri_ok")
)

cartoes_com_ri = cartoes_tipado.join(contas_silver, "id_conta", "left")

cartoes_avaliado = cartoes_com_ri.withColumn(
    "motivo_erro",
    F.concat_ws(
        " | ",
        F.when(F.col("id_cartao").isNull(), F.lit("chave_nula:id_cartao")),
        F.when(F.col("id_conta").isNull(), F.lit("chave_nula:id_conta")),
        F.when(F.col("data_emissao").isNull(), F.lit("data_emissao_invalida")),
        F.when(F.col("_ri_ok").isNull(), F.lit("integridade_referencial:id_conta_inexistente")),
    ),
).select(*[F.col(c) for c in COLUNAS_CARTOES], F.col("motivo_erro"))

cartoes_validos = cartoes_avaliado.filter(F.col("motivo_erro") == "")

qtd_q_cartoes = salvar_quarentena(cartoes_avaliado, COLUNAS_CARTOES, f"{SILVER}.quarentena_cartoes")
qtd_s_cartoes = publicar_silver(
    cartoes_validos,
    COLUNAS_CARTOES,
    f"{SILVER}.cartoes",
    "id_cartao",
    [F.col("data_atualizacao").desc(), F.col("id_conta").desc()],
)
print(f"CARTÕES: bronze={cartoes_tipado.count()} | silver={qtd_s_cartoes} | quarentena={qtd_q_cartoes}")

In [0]:
# ============================================================
# 4. TRANSAÇÕES — tipagem, validação, RI e deduplicação
# Regras: valor deve ser numérico positivo (decimal(18,2));
#         id_cartao deve existir em silver.cartoes (cascata);
#         dedup por id_transacao (não há data_atualizacao;
#         usa data_transacao/valor como ordenação determinística).
# ============================================================
COLUNAS_TRANSACOES = ["id_transacao", "id_cartao", "data_transacao", "valor", "nome_estabelecimento", "mcc", "cidade_estabelecimento"]

transacoes_tipado = spark.table(f"{BRONZE}.transacoes_cartao").select(
    F.col("id_transacao").cast("long").alias("id_transacao"),
    F.col("id_cartao").cast("long").alias("id_cartao"),
    F.expr("try_cast(data_transacao as date)").alias("data_transacao"),
    F.expr("try_cast(valor as decimal(18,2))").alias("valor"),
    F.col("nome_estabelecimento").cast("string").alias("nome_estabelecimento"),
    F.expr("try_cast(mcc as int)").alias("mcc"),
    F.col("cidade_estabelecimento").cast("string").alias("cidade_estabelecimento"),
)

cartoes_silver = spark.table(f"{SILVER}.cartoes").select(
    F.col("id_cartao"), F.lit(True).alias("_ri_ok")
)

transacoes_com_ri = transacoes_tipado.join(cartoes_silver, "id_cartao", "left")

transacoes_avaliado = transacoes_com_ri.withColumn(
    "motivo_erro",
    F.concat_ws(
        " | ",
        F.when(F.col("id_transacao").isNull(), F.lit("chave_nula:id_transacao")),
        F.when(F.col("id_cartao").isNull(), F.lit("chave_nula:id_cartao")),
        F.when(F.col("data_transacao").isNull(), F.lit("data_transacao_invalida")),
        F.when(F.col("valor").isNull() | (F.col("valor") <= 0), F.lit("valor_invalido")),
        F.when(F.col("_ri_ok").isNull(), F.lit("integridade_referencial:id_cartao_inexistente")),
    ),
).select(*[F.col(c) for c in COLUNAS_TRANSACOES], F.col("motivo_erro"))

transacoes_validos = transacoes_avaliado.filter(F.col("motivo_erro") == "")

qtd_q_trans = salvar_quarentena(
    transacoes_avaliado, COLUNAS_TRANSACOES, f"{SILVER}.quarentena_transacoes_cartao"
)
qtd_s_trans = publicar_silver(
    transacoes_validos,
    COLUNAS_TRANSACOES,
    f"{SILVER}.transacoes_cartao",
    "id_transacao",
    [F.col("data_transacao").desc(), F.col("valor").desc(), F.col("id_cartao").desc()],
)
print(f"TRANSAÇÕES: bronze={transacoes_tipado.count()} | silver={qtd_s_trans} | quarentena={qtd_q_trans}")

In [0]:
# ============================================================
# 5. ESTORNOS + FLAG DE ESTORNO
# Regras: id_transacao deve existir em silver.transacoes_cartao;
#         transações estornadas recebem flag_estornada = true e
#         são EXCLUÍDAS de silver.transacoes_financeiras, evitando
#         contaminação dos indicadores financeiros da Gold.
# ============================================================
COLUNAS_ESTORNOS = ["id_estorno", "id_transacao", "data_estorno", "motivo_estorno"]

estornos_tipado = spark.table(f"{BRONZE}.estornos").select(
    F.col("id_estorno").cast("long").alias("id_estorno"),
    F.col("id_transacao").cast("long").alias("id_transacao"),
    F.expr("try_cast(data_estorno as date)").alias("data_estorno"),
    F.col("motivo_estorno").cast("string").alias("motivo_estorno"),
)

transacoes_silver = spark.table(f"{SILVER}.transacoes_cartao").select(
    F.col("id_transacao"), F.lit(True).alias("_ri_ok")
)

estornos_com_ri = estornos_tipado.join(transacoes_silver, "id_transacao", "left")

estornos_avaliado = estornos_com_ri.withColumn(
    "motivo_erro",
    F.concat_ws(
        " | ",
        F.when(F.col("id_estorno").isNull(), F.lit("chave_nula:id_estorno")),
        F.when(F.col("id_transacao").isNull(), F.lit("chave_nula:id_transacao")),
        F.when(F.col("data_estorno").isNull(), F.lit("data_estorno_invalida")),
        F.when(F.col("_ri_ok").isNull(), F.lit("integridade_referencial:id_transacao_inexistente")),
    ),
).select(*[F.col(c) for c in COLUNAS_ESTORNOS], F.col("motivo_erro"))

estornos_validos = estornos_avaliado.filter(F.col("motivo_erro") == "")

qtd_q_est = salvar_quarentena(estornos_avaliado, COLUNAS_ESTORNOS, f"{SILVER}.quarentena_estornos")
qtd_s_est = publicar_silver(
    estornos_validos,
    COLUNAS_ESTORNOS,
    f"{SILVER}.estornos",
    "id_estorno",
    [F.col("data_estorno").desc(), F.col("id_transacao").desc()],
)
print(f"ESTORNOS: bronze={estornos_tipado.count()} | silver={qtd_s_est} | quarentena={qtd_q_est}")

# --- Flag de transação estornada em silver.transacoes_cartao ---
# distinct: uma transação pode ter mais de um estorno; sem o distinct o
# join multiplicaria as linhas de transações.
ids_estornados = (
    spark.table(f"{SILVER}.estornos")
    .select("id_transacao")
    .distinct()
    .withColumn("_estornada", F.lit(True))
)

transacoes_flag = (
    spark.table(f"{SILVER}.transacoes_cartao")
    .join(ids_estornados, "id_transacao", "left")
    .withColumn("flag_estornada", F.coalesce(F.col("_estornada"), F.lit(False)))
    .drop("_estornada")
)
transacoes_flag.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SILVER}.transacoes_cartao"
)

# --- Base financeira SEM transações estornadas (insumo da Gold) ---
(
    spark.table(f"{SILVER}.transacoes_cartao")
    .filter(~F.col("flag_estornada"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{SILVER}.transacoes_financeiras"
    )
)

total_trans = spark.table(f"{SILVER}.transacoes_cartao").count()
total_estornadas = spark.table(f"{SILVER}.transacoes_cartao").filter(F.col("flag_estornada")).count()
total_financeiras = spark.table(f"{SILVER}.transacoes_financeiras").count()
print(f"Transações silver={total_trans} | estornadas={total_estornadas} | financeiras (sem estorno)={total_financeiras}")

In [0]:
# ============================================================
# 6. RESUMO DO PIPELINE SILVER — qualidade e quarentena
# ============================================================
resumo = [
    ("clientes", f"{BRONZE}.clientes", f"{SILVER}.clientes", f"{SILVER}.quarentena_clientes"),
    ("contas", f"{BRONZE}.contas", f"{SILVER}.contas", f"{SILVER}.quarentena_contas"),
    ("cartoes", f"{BRONZE}.cartoes", f"{SILVER}.cartoes", f"{SILVER}.quarentena_cartoes"),
    ("transacoes_cartao", f"{BRONZE}.transacoes_cartao", f"{SILVER}.transacoes_cartao", f"{SILVER}.quarentena_transacoes_cartao"),
    ("estornos", f"{BRONZE}.estornos", f"{SILVER}.estornos", f"{SILVER}.quarentena_estornos"),
]

linhas = []
for entidade, t_bronze, t_silver, t_quar in resumo:
    bronze_n = spark.table(t_bronze).count()
    silver_n = spark.table(t_silver).count()
    quar_n = spark.table(t_quar).count()
    pct = round(100.0 * quar_n / bronze_n, 2) if bronze_n else 0.0
    linhas.append((entidade, bronze_n, silver_n, quar_n, pct))

df_resumo = spark.createDataFrame(
    linhas,
    "entidade string, bronze long, silver long, quarentena long, pct_quarentena double",
)
display(df_resumo)

# Distribuição dos motivos de erro nas tabelas de quarentena
motivos = []
for entidade, _, _, t_quar in resumo:
    m = (
        spark.table(t_quar)
        .groupBy("motivo_erro")
        .count()
        .withColumn("entidade", F.lit(entidade))
        .select("entidade", "motivo_erro", "count")
    )
    motivos.append(m)

df_motivos = motivos[0]
for m in motivos[1:]:
    df_motivos = df_motivos.unionByName(m)

display(df_motivos.orderBy("entidade", F.col("count").desc()))